# Loading OOD experiments results.

## Preparing the notebook.

### Importing the libraries.

In [1]:
import os
import sys

from anndata import AnnData
import hydra
import matplotlib.pyplot as plt
import numpy as np
from omegaconf import OmegaConf
import pandas as pd
import scanpy as sc
import torch
from tqdm import tqdm
import wandb

from sc_exp_design import constants 
from sc_exp_design.training.callbacks import MetricsCallBack
from sc_exp_design.models import FlowMatching

sys.path.insert(0, "../../../../scripts")
from data_utils import annotate_perturbations, apply_shared_transformations
from ood_utils import split_adata
from validation_utils import predict_on_ood_data

### Constants.

In [2]:
TEAM = "inverse-perturbation-models"
PROJECT = "haematopoiesis_forward_model_ohe_protocols_X_channel_concat"

## Load wandb results.

### Prepare API and load runs.

In [3]:
api = wandb.Api()
runs = api.runs(f"{TEAM}/{PROJECT}")

### Retrieving example config for access to shared settings.

In [4]:
first_config = runs[0].config
first_config = OmegaConf.create(first_config)
OmegaConf.resolve(first_config)

## Data.

### Reading data.

In [5]:
h5ad_path = first_config["paths"]["h5ad_path"]
adata = sc.read_h5ad(h5ad_path)
adata

AnnData object with n_obs × n_vars = 93275454 × 21
    obs: 'replicate', 'date', 'cytometer', 'cytometer_serial_no', 'well_id', 'count_beads', 'cell_counts', 'experiment_number', 'experiment_id', '740-yp_[µm]', 'mtg_[µm]', 'rhflt3l_[ng_ml]', 'gm-csf_[ng_ml]', 'tpo_[ng_ml]', 'sr1_[nm]', 'um171_[nm]', 'um729_[µm]', 'scf_[ng_ml]', 'butyzamide_[nm]', 'retinoic_acid_[µm]', 'ldl_[ng_ml]', 'il3_[ng_ml]', 'source_id', 'hydrogel_type', 'o2_[%]', 'source_cell_phenotype', 'total_count_beads', 'counts_source_cell_phenotype', 'days_of_culture', 'plate', 'absolute_number_multiplier_[2^n]', 'cytometer_id', 'FSC - Area', 'FSC - Height', 'FSC - Width', 'SSC - Area', 'SSC - Height', 'SSC - Width', 'Time Stamp', 'batch'
    var: 'n', 'channel', 'marker', '$PnD', '$PnB', '$PnR', '$PnG', '$PnE', 'signal_type', 'antibody'
    uns: 'meta', 'processing_steps'
    layers: 'positives', 'raw'

### Annotating perturbations.

In [6]:
column2tranform = {}
adata = annotate_perturbations(
    adata,
    first_config.annotation.protocol_columns,
    protocol_obs_key_added=first_config.annotation.protocol_obs_key_added,
    protocol_sep=first_config.annotation.protocol_sep,
    one_hot_uns_key_added=first_config.annotation.one_hot_uns_key_added,
    column2tranform=column2tranform,
    protocol_obsm_key=first_config.annotation.protocol_obsm_key,
)

## Preparing evaluation.

### Defining utility functions.

In [7]:
def init_and_resolve_omegaconf(
    config_dict,
):
    cfg = OmegaConf.create(config_dict)
    OmegaConf.resolve(cfg)
    return cfg

def load_model_from_run(
    run_name,
    config
):
    dump_dir = config.paths.dump_dir
    dump_path = os.path.join(dump_dir, f"{run_name}_FlowMatching.pkl")
    return FlowMatching.load(dump_path)
    
def get_ood_splits_and_compute_shared_transformations(
    adata,
    config
):
    train_adata, ood_adatas_dict = split_adata(
        adata,
        config.ood.obs_column,
        config.ood.unique_value_ids,
    )
    train_adata, ood_adatas_dict = apply_shared_transformations(
        train_adata,
        ood_adatas_dict,
        config.transforms.scatter_columns,
        scatter_obsm_key=config.transforms.scatter_obsm_key,
        channel_concat_obsm_key=config.transforms.channel_concat_obsm_key,
        pca_obsm_key=config.transforms.pca_obsm_key,
        pca_concat_obsm_key=config.transforms.pca_concat_obsm_key,
        standardize_repr=config.transforms.standardize_repr,
        sample_rep=config.transforms.sample_rep,
        sample_rep_obsm_key=config.transforms.sample_rep_obsm_key,
        repr_params_uns_key=config.transforms.repr_params_uns_key,
        compute_channel_pcs=config.transforms.compute_channel_pcs,
        standardize_channel_feats=config.transforms.standardize_channel_feats,
        channel_feats_obsm_key=config.transforms.channel_feats_obsm_key,
        channel_params_uns_key=config.transforms.channel_params_uns_key,
        standardize_scatter_feats=config.transforms.standardize_scatter_feats,
        scatter_params_uns_key=config.transforms.scatter_params_uns_key,
    )
    return train_adata, ood_adatas_dict

def predict_with_run(
    adata,
    run,
    N=10_000,
    num_time_steps=100,
    solver_kwargs={}
):
    # resolving config
    config = init_and_resolve_omegaconf(run.config)
    # loading model
    print(f"Loading model for run {run.name}")
    flow_matching = load_model_from_run(run.name, config)
    print(f"Splitting data for {run.name}")
    # retrieving ood splits
    train_adata, ood_adatas_dict = get_ood_splits_and_compute_shared_transformations(adata, config)
    print(f"{run.name}")
    print(f"train data {train_adata.shape}")
    for k, ood_adata in ood_adatas_dict.items():
        print(f"{k}-ood data {ood_adata.shape}")

    # preparing validation data
    ood_data_dict = {
        k: flow_matching.data_manager.get_data(v) for k, v in ood_adatas_dict.items()
    }
    return {
        k: predict_on_ood_data(
            N,
            flow_matching,
            ood_data,
            num_time_steps,
            solver_kwargs,
        ) for k, ood_data in ood_data_dict.items()
    }

### Initializing metrics.

In [8]:
metrics_callback = MetricsCallBack(
    ["r_squared", "energy_distance", ],
    state_transforms=None,
)

### Iterating over the runs and saving the model.

In [10]:
N=10_000,
num_time_steps=100,
solver_kwargs={"solver":"euler"}

# storing the prediction for each ood column for easier comparison
# format
# ||ood_column
# |--|ood_value
# |--|--|run_name
# |--|--|--|results_dict
ood_preds_per_column = {}
for run in tqdm(runs):
    run_name = run.name
    if run.state in ["killed", "failed"]:
        print(f"Run {run_name} was killed.")
        continue
    if run.state == "running":
        print(f"Run {run_name} still running.")
        continue
    # creating keys for current ood settings
    obs_col = run.config["ood"]["obs_column"]
    ood_val = run.config["ood"]["unique_value_ids"]
    if obs_col != 'gm-csf_[ng_ml]':
        continue
    if ood_val != 1:
        continue
    print(f"Computing results for {run_name}")
    print(f"OOD column {obs_col} with unique val {ood_val}")
    if obs_col not in ood_preds_per_column:
        ood_preds_per_column[obs_col] = {}
    if ood_val not in ood_preds_per_column[obs_col]:
        ood_preds_per_column[obs_col][ood_val] = {}

    # storing results for run name
    ood_preds_per_column[obs_col][ood_val][run_name] = predict_with_run(
        adata,
        run,
        N=10_000,
        num_time_steps=100,
        solver_kwargs={"solver":"euler"}
    )
    break

  0%|          | 0/160 [00:00<?, ?it/s]

Run genial-sea-1 was killed.
Run floral-night-2 was killed.
Run zany-bush-3 was killed.
Computing results for robust-brook-5
OOD column gm-csf_[ng_ml] with unique val 1
Loading model for run robust-brook-5
Splitting data for robust-brook-5


: 